# PSMAP surrogate example

This notebook demonstrates `aispy.psmap.PSMAPSurrogate`: a fast surrogate
for ais++ simulations built from a precomputed phase space map.

### What a PSMAP contains

Running ais++ with `initmode psgrid` produces a `_PSMAP.h5` file with one
row per output port per grid atom:

| field | meaning |
|---|---|
| `initial_positions/velocities` | $(x_0, v_{x0})$ of the atom |
| `amp0`, `amp1` | wavepacket amplitudes $A_0$, $A_1$ |
| `phase_shifts` | $\Delta\phi$ (quad-precise from C++) |
| `final_positions/velocities` | mean port position at detection |
| `states`, `is_interfering` | port metadata |

### How the surrogate works

1. **Interpolate** $\Delta\phi$, $A_0$, $A_1$ at arbitrary $(x_0, v_{x0})$
   using `RegularGridInterpolator`.
2. **Map** initial to final coordinates analytically:
   $x_f = x_0 + v_{x0}\,t_\mathrm{det}$, $v_{xf} = v_{x0}$
   (exact for free transverse motion).
3. **Apply** an arbitrary phase profile $\Phi(x_f, v_{xf})$ on top of
   $\Delta\phi + \phi_0$.
4. **Sample** each atom's output state: $\mathrm{Bernoulli}(P_{s=0})$,
   giving realistic Poisson shot noise.

A surrogate shot with $10^5$ atoms runs in $\sim$10 ms versus several
minutes for a full ais++ run.

### Setup

```bash
pip install aispy   # or: pip install -e /path/to/aispy

# Generate the PSMAP first (from the aispp repo):
cd /path/to/aispp/examples
ais++ -i input-files/PSR_EXAMPLE_PSGRID.aisi -o output-files/PSR_EXAMPLE_PSGRID.h5
```
Then set `PSMAP_FILE` below to the path of that `.h5` file.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

import sys
sys.path.insert(0, '..')   # use local aispy if not installed
from aispy.psmap import load_psmap, PSMAPSurrogate

try:
    plt.style.use('seaborn-v0_8-ticks')
except OSError:
    plt.style.use('seaborn-ticks')
plt.rcParams.update({
    'font.family': 'serif', 'font.size': 9,
    'axes.labelsize': 9, 'savefig.dpi': 150,
    'xtick.direction': 'in', 'ytick.direction': 'in',
    'legend.frameon': False,
})

# ── path to the PSMAP produced by ais++ psgrid run ───────────────────────────
PSMAP_FILE = '../../aispp/examples/output-files/PSR_EXAMPLE_PSGRID.h5'
T_DET      = 4.451          # detection time [s]  (from PSR_EXAMPLE_PSGRID.aisi)

# ── reference cloud parameters (same as the PSR_EXAMPLE_NLMT1 Gaussian run) ─
SIGMA_X    = 1e-4           # 100 µm position spread
T_TRANS    = 1e-9           # 1 nK transverse temperature
kB         = 1.380649e-23
M_SR87     = 86.909 * 1.66054e-27
SIGMA_VX   = np.sqrt(kB * T_TRANS / M_SR87)   # ≈ 3.1e-4 m/s
print(f'σ_x  = {SIGMA_X*1e6:.0f} µm,  σ_vx = {SIGMA_VX*1e3:.2f} mm/s')

## 1. Load the PSMAP and build the surrogate

In [ ]:
data = load_psmap(PSMAP_FILE)
sur  = PSMAPSurrogate(data, t_det=T_DET)

print(f'Grid          : {sur.nx} × {sur.nvx}')
print(f'x0 range      : {sur.xs[0]*1e6:.0f} … {sur.xs[-1]*1e6:.0f} µm')
print(f'vx0 range     : {sur.vxs[0]*1e3:.2f} … {sur.vxs[-1]*1e3:.2f} mm/s')
print(f'Ports / atom  : {sur.nP}')
for pi in range(sur.nP):
    tag = 'interfering' if sur.port_interfering[pi] else 'solo'
    print(f'  port {pi}  state={sur.port_states[pi]}  '
          f'{sur.port_path0[pi]}/{sur.port_path1[pi]}  {tag}')

## 2. Fit quality: how well does a quadratic model describe Δφ?

For a Gaussian beam the analytical prediction (Eq. 11 of Mouelle et al. 2025)
is a quadratic in $(x_0, v_{x0})$.  A high $R^2$ confirms the grid captures
the wavefront accurately.

In [ ]:
fq = sur.fit_quality()
print('Quadratic fit quality per port:')
for pi, m in fq.items():
    print(f'  port {pi}  R²={m["r2"]:.6f}  '
          f'residual_std={m["residual_std_rad"]*1e3:.3f} mrad')

## 3. Δφ and amplitude maps from the surrogate

Evaluate the surrogate on the original grid to reproduce the raw PSMAP maps.
The Δφ residual (after removing the dominant linear terms) reveals the
quadratic wavefront curvature.

In [ ]:
# Evaluate on the original grid
X0g, VX0g = np.meshgrid(sur.xs, sur.vxs, indexing='ij')   # (nx, nvx)
dphi_g, amp0_g, amp1_g = sur.eval(X0g.ravel(), VX0g.ravel())
dphi_g  = dphi_g.reshape(sur.nx, sur.nvx, sur.nP)
amp0_g  = amp0_g.reshape(sur.nx, sur.nvx, sur.nP)
amp1_g  = amp1_g.reshape(sur.nx, sur.nvx, sur.nP)

# Remove mean from Δφ for display
dphi_res = dphi_g - dphi_g.mean(axis=(0,1), keepdims=True)

extent = [sur.vxs[0]*1e3, sur.vxs[-1]*1e3, sur.xs[0]*1e6, sur.xs[-1]*1e6]
fig, axes = plt.subplots(2, sur.nP, figsize=(3*sur.nP, 5),
                         gridspec_kw=dict(hspace=0.45, wspace=0.4))
if sur.nP == 1:
    axes = axes[:, np.newaxis]

for pi in range(sur.nP):
    lbl = f'{sur.port_path0[pi]}/{sur.port_path1[pi]}'
    vabs = max(np.abs(dphi_res[:,:,pi]).max(), 1e-9)
    im0 = axes[0,pi].imshow(dphi_res[:,:,pi], origin='lower', extent=extent,
                             aspect='auto', cmap='RdBu', vmin=-vabs, vmax=vabs)
    axes[0,pi].set_title(f'{lbl}  (state {sur.port_states[pi]})', fontsize=8)
    axes[0,pi].set_xlabel(r'$v_{x0}$ [mm/s]')
    if pi == 0: axes[0,pi].set_ylabel(r'$x_0$ [µm]')
    plt.colorbar(im0, ax=axes[0,pi], label=r'$\Delta\phi-\langle\Delta\phi\rangle$ [rad]')

    avg_amp = (amp0_g[:,:,pi] + amp1_g[:,:,pi]) / (1 + sur.port_interfering[pi])
    im1 = axes[1,pi].imshow(avg_amp, origin='lower', extent=extent,
                             aspect='auto', cmap='plasma', vmin=0)
    axes[1,pi].set_xlabel(r'$v_{x0}$ [mm/s]')
    if pi == 0: axes[1,pi].set_ylabel(r'$x_0$ [µm]')
    plt.colorbar(im1, ax=axes[1,pi], label=r'mean $|c|$')

axes[0,0].annotate(r'$\Delta\phi$ residual', xy=(-0.4,0.5), xycoords='axes fraction',
                   fontsize=8, fontweight='bold', rotation=90, va='center')
axes[1,0].annotate('mean amplitude', xy=(-0.4,0.5), xycoords='axes fraction',
                   fontsize=8, fontweight='bold', rotation=90, va='center')
plt.suptitle('Surrogate maps (evaluated on original grid)', y=1.01)
plt.show()

## 4. Generate a single synthetic shot

Sample $N$ atoms from the Gaussian cloud, propagate through the surrogate,
and histogram their final transverse position $x_f = x_0 + v_{x0}\,t_\mathrm{det}$.

With no phase shear the two output states are not spatially separated, so
both the ground- and excited-state histograms show the Gaussian cloud
envelope.

In [ ]:
N_ATOMS = 300_000

df = sur.generate_atoms(
    mu_x0=0., mu_vx0=0., sigma_x=SIGMA_X, sigma_vx=SIGMA_VX,
    phi0=0., natoms=N_ATOMS, rng=42)

print(f'Atoms generated   : {len(df):,}')
print(f'Ground state (s=0): {(df.state==0).sum():,}  '
      f'({(df.state==0).mean()*100:.1f}%)')
print(f'Excited state (s=1): {(df.state==1).sum():,}')
print(f'xf range          : {df.xf.min()*1e3:.1f} … {df.xf.max()*1e3:.1f} mm')

bins   = np.linspace(-8e-3, 8e-3, 80)
xc     = 0.5*(bins[:-1]+bins[1:])
bw_mm  = (xc[1]-xc[0])*1e3

h0, _ = np.histogram(df.xf[df.state==0], bins=bins)
h1, _ = np.histogram(df.xf[df.state==1], bins=bins)

fig, ax = plt.subplots(figsize=(5.5, 3))
ax.bar(xc*1e3, h0, width=bw_mm, color='#2ca02c', alpha=0.7, label='state 0')
ax.bar(xc*1e3, h1, width=bw_mm, color='#d62728', alpha=0.5, label='state 1')
ax.set_xlabel(r'Final $x_f$ [mm]')
ax.set_ylabel('Atom count')
ax.set_title(f'Single shot, no phase shear ($N={N_ATOMS:,}$, $\\phi_0=0$)')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Offline phase shear — PSR fringe

Applying $\Phi(x_f) = \kappa\,x_f$ modifies the port probability:
$$P(x_f,\kappa) = A_0^2+A_1^2+2A_0A_1\cos\bigl(\Delta\phi(x_0,v_{x0})+\kappa\,x_f\bigr)$$
Since $x_f = x_0 + v_{x0}\,t_\mathrm{det}$ is a smooth injective map of the
initial conditions, atoms at different $x_f$ positions receive different
phase boosts — creating spatial fringes in the histogram.

In [ ]:
def fringe_model(x, A, sigma, mu, kappa_fit, phi0, C):
    return A*(1+C*np.cos(kappa_fit*x+phi0))*np.exp(-(x-mu)**2/(2*sigma**2))

kappa_vals = [500, 1500, 3140, 6280]
colors     = ['#1f77b4','#ff7f0e','#2ca02c','#d62728']

fig, axes = plt.subplots(len(kappa_vals), 1, figsize=(5.5, 7),
                         sharex=True, gridspec_kw=dict(hspace=0.08))

for ax, kappa, col in zip(axes, kappa_vals, colors):
    df_k = sur.generate_atoms(
        mu_x0=0., mu_vx0=0., sigma_x=SIGMA_X, sigma_vx=SIGMA_VX,
        phi0=0., natoms=N_ATOMS,
        phase_profile=lambda xf, vxf: kappa * xf,
        rng=42)

    h0k, _ = np.histogram(df_k.xf[df_k.state==0], bins=bins)
    ax.bar(xc*1e3, h0k, width=bw_mm, color=col, alpha=0.55, linewidth=0)

    try:
        p0 = [h0k.max(), df_k.xf.std(), df_k.xf[df_k.state==0].mean(), kappa, 0., 0.5]
        bounds = ([0,0,xc.min(),kappa*0.5,-np.pi,0.],
                  [np.inf,0.02,xc.max(),kappa*2., np.pi,1.])
        popt, _ = curve_fit(fringe_model, xc, h0k, p0=p0, bounds=bounds, maxfev=8000)
        ax.plot(np.linspace(xc[0],xc[-1],500)*1e3,
                fringe_model(np.linspace(xc[0],xc[-1],500), *popt),
                'k-', lw=1.2)
        ax.annotate(
            fr'$\kappa={kappa}\,\rm rad/m$  $C={popt[5]:.2f}$  '
            fr'$\hat\kappa={popt[3]:.0f}\,\rm rad/m$',
            xy=(0.02,0.84), xycoords='axes fraction', fontsize=8)
    except RuntimeError:
        ax.annotate(fr'$\kappa={kappa}\,\rm rad/m$ (fit failed)',
                    xy=(0.02,0.84), xycoords='axes fraction', fontsize=8)

    ax.set_ylabel('Counts', fontsize=8)

axes[-1].set_xlabel(r'Final $x_f$ [mm]')
plt.suptitle(f'Offline PSR fringe for different shear wave-vectors ($N={N_ATOMS:,}$)',
             y=1.01)
plt.show()

## 6. Phase scan with shot noise

Scan $\phi_0$ over one full fringe and record the ground-state fraction
$N_g/(N_g+N_e)$ for each shot.  Unlike the deterministic weighted-sum approach
in the aispp notebook, each shot here has realistic Poisson fluctuations.

In [ ]:
KAPPA_PSR  = 3140.           # rad/m — shear applied for readout
N_PHI_SCAN = 24
N_SHOT     = 50_000          # atoms per shot (smaller for speed)

phi0_vals  = np.linspace(0, 2*np.pi, N_PHI_SCAN, endpoint=False)
fg_vals    = np.zeros(N_PHI_SCAN)   # ground-state fraction per shot

rng = np.random.default_rng(0)
for k, phi0 in enumerate(phi0_vals):
    df_k = sur.generate_atoms(
        mu_x0=0., mu_vx0=0., sigma_x=SIGMA_X, sigma_vx=SIGMA_VX,
        phi0=phi0, natoms=N_SHOT,
        phase_profile=lambda xf, vxf: KAPPA_PSR * xf,
        rng=rng)
    fg_vals[k] = (df_k.state == 0).mean()

# Fit sinusoid
def fringe_1d(phi, A, C, phi_off):
    return A * (1 + C*np.cos(phi + phi_off))

popt, _ = curve_fit(fringe_1d, phi0_vals, fg_vals,
                    p0=[fg_vals.mean(), 0.8, 0.0])

shot_noise = 1. / (2*np.sqrt(N_SHOT))   # ideal σ for C=1

fig, ax = plt.subplots(figsize=(5.5, 3))
phi_fine = np.linspace(0, 2*np.pi, 300)
ax.plot(phi_fine/np.pi, fringe_1d(phi_fine, *popt), 'k-', lw=1.2, label='fit')
ax.errorbar(phi0_vals/np.pi, fg_vals, yerr=shot_noise,
            fmt='o', ms=4, color='#2ca02c', label='surrogate shots')
ax.set_xlabel(r'$\phi_0 / \pi$')
ax.set_ylabel(r'$N_g / N_{\rm tot}$')
ax.set_title(f'Phase scan  ($N={N_SHOT:,}$/shot,  $\\kappa={KAPPA_PSR:.0f}\,\\rm rad/m$)'
             f'  contrast={popt[1]:.3f}')
ax.set_xlim(0, 2)
ax.legend()
plt.tight_layout()
plt.show()

print(f'Fitted contrast : {popt[1]:.4f}')
print(f'Ideal shot noise: {shot_noise:.4f}  '
      f'(observed scatter ≈ {fg_vals.std():.4f})')

## 7. Cloud-parameter scan — no ais++ calls

Scan the cloud centre-of-mass position $\mu_{x0}$ and record the measured
phase from a single surrogate shot per point.  This mimics a systematic
bias study that would require hundreds of full ais++ runs in the conventional
approach.

In [ ]:
from scipy.optimize import curve_fit as _cf

def measure_phase(mu_x0, mu_vx0=0., phi0_scan=None, **cloud_kw):
    """Estimate interferometer phase for a given cloud COM."""
    if phi0_scan is None:
        phi0_scan = np.linspace(0, 2*np.pi, 8, endpoint=False)
    fg = np.array([
        sur.generate_atoms(
            mu_x0=mu_x0, mu_vx0=mu_vx0,
            phi0=phi0, natoms=30_000,
            phase_profile=lambda xf, vxf: KAPPA_PSR * xf,
            rng=rng, **cloud_kw
        )['state'].eq(0).mean()
        for phi0 in phi0_scan
    ])
    popt, _ = _cf(fringe_1d, phi0_scan, fg, p0=[fg.mean(), 0.7, 0.0])
    return float(popt[2])   # phase offset

mu_x0_scan = np.linspace(-2e-4, 2e-4, 20)
rng = np.random.default_rng(1)
phases_meas = np.array([
    measure_phase(mx, sigma_x=SIGMA_X, sigma_vx=SIGMA_VX)
    for mx in mu_x0_scan
])
# Unwrap
phases_meas = np.unwrap(phases_meas)

fig, ax = plt.subplots(figsize=(5.5, 3))
ax.plot(mu_x0_scan*1e6, np.rad2deg(phases_meas - phases_meas.mean()), 'o-', ms=4)
ax.set_xlabel(r'Cloud COM $\mu_{x0}$ [µm]')
ax.set_ylabel(r'Phase bias $\delta\hat{\phi}$ [deg]')
ax.set_title('Phase bias vs cloud position — surrogate only, zero ais++ calls')
plt.tight_layout()
plt.show()

## 8. Speed comparison

In [ ]:
import time

natoms_list = [10_000, 50_000, 100_000, 500_000, 1_000_000]
times = []
for n in natoms_list:
    t0 = time.perf_counter()
    sur.generate_atoms(0., 0., SIGMA_X, SIGMA_VX,
                       phase_profile=lambda xf, vxf: KAPPA_PSR*xf,
                       natoms=n, rng=0)
    times.append(time.perf_counter() - t0)

fig, ax = plt.subplots(figsize=(4.5, 3))
ax.loglog(natoms_list, times, 'o-', ms=5)
ax.set_xlabel('Atoms per shot')
ax.set_ylabel('Wall time [s]')
ax.set_title('Surrogate throughput')
ax.grid(True, which='both', alpha=0.3)
for n, t in zip(natoms_list, times):
    ax.annotate(f'{t*1e3:.0f} ms', (n, t), textcoords='offset points',
                xytext=(4, 2), fontsize=7)
plt.tight_layout()
plt.show()

print('Surrogate timing:')
for n, t in zip(natoms_list, times):
    print(f'  {n:>8,} atoms  →  {t*1e3:6.1f} ms')